# 관통 프로젝트 5단계 (풀이)

검증용 풀이 노트북입니다. 공유하지 않습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install pypdf beautifulsoup4 transformers
# 13장과 관통 프로젝트 5단계에서 씁니다. 예제 13-1은 1장의 토크나이저만 씁니다(모델은 내려받지 않습니다)
import embed
from doc_tools import CATALOG, TOOLS, read_doc, search_docs   # 사내 문서 120건의 검색과 읽기, 도구 정의(노트북/doc_tools.py)

LOCAL_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
embed.ask_endpoint(LOCAL_MODEL)          # 토크나이저가 PC에 없고 HF_ENDPOINT가 비어 있으면 주소를 묻습니다
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)


# 관통 프로젝트 1, 2단계의 풀이가 쓰는 것입니다(5장, 6장의 준비 코드와 같습니다)
import csv
import random

with open(f'{DATA}/문의.csv', encoding='utf-8') as file:
    inquiries = list(csv.DictReader(file))        # 문의 60건. 열: 번호, 문의, 라벨, 긴급, 비고
LABELS = ['설비 이상', '품질', '안전', '문서와 데이터 요청']
PRICES = {'소형': {'input': 0.10, 'output': 0.50}, '기본': {'input': 2.00, 'output': 10.00},
          '대형': {'input': 4.00, 'output': 20.00}}   # 100만 토큰당 단가(달러). 5장과 같습니다(2026-11 확인)


def cost(reply, tier):
    """응답 하나의 요금(달러). 입력 토큰과 출력 토큰(thinking 포함)에 각각의 단가를 곱해 더합니다"""
    price = PRICES[tier]
    return (reply.input_tokens * price['input'] + reply.output_tokens * price['output']) / 1_000_000


print(f'문서 {len(CATALOG)}건, 도구 {[tool["name"] for tool in TOOLS]}, 토크나이저 {LOCAL_MODEL}')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


문서 120건, 도구 ['search_docs', 'read_doc'], 토크나이저 Qwen/Qwen2.5-0.5B-Instruct
소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 앞 단계의 코드: 1단계 풀이

In [3]:
SYSTEM = '너는 반도체 후공정 라인의 문의에 답하는 도우미다. 사내 정보를 모르면 지어내지 말고 모른다고 답한다.'


def answer(question, tier='소형', tries=3, send=llm.send, sleep=time.sleep):
    """질문 하나에 답합니다. 돌려주는 dict의 키: question, status('완료', '확인 필요', '실패'),
    answer, input_tokens, output_tokens, cost, reason"""
    result = {'question': question, 'status': '실패', 'answer': '', 'input_tokens': 0,
              'output_tokens': 0, 'cost': 0.0, 'reason': ''}
    for attempt in range(tries):
        try:
            reply = send([{'role': 'user', 'content': question}], system=SYSTEM, model=tier, max_tokens=2000)
        except llm.CallError as error:
            result['reason'] = str(error)
            if not error.retryable:
                return result                         # 요청을 고쳐야 풀리는 오류는 다시 보내지 않습니다
            if attempt < tries - 1:
                sleep(2 ** attempt + random.random())  # 1초, 2초 … 백오프에 지터를 더합니다
            continue
        result.update(answer=reply.text, input_tokens=reply.input_tokens, output_tokens=reply.output_tokens,
                      cost=cost(reply, tier), reason=reply.stop_reason,
                      status='완료' if reply.stop_reason == 'end' else '확인 필요')
        return result
    return result


def mock_send(results):
    """미리 정해 둔 결과를 차례로 돌려주는 mock. 결과가 예외이면 발생시킵니다"""
    queue = list(results)

    def send(messages, **options):
        result = queue.pop(0)
        if isinstance(result, Exception):
            raise result
        return result
    return send


done = llm.Reply(text='완료된 답', input_tokens=50, output_tokens=20, stop_reason='end', model='mock', seconds=0.1)
cut = llm.Reply(text='잘린 답', input_tokens=50, output_tokens=2000, stop_reason='max_tokens', model='mock', seconds=0.1)
busy, missing = llm.CallError('rate_limit', 429, '요청이 많음'), llm.CallError('not_found', 404, '없는 모델')
no_wait = lambda seconds: None                       # 테스트에서는 기다리지 않습니다
print('429 두 번 뒤 성공:', answer('질문', send=mock_send([busy, busy, done]), sleep=no_wait)['status'])
print('429 세 번:', answer('질문', send=mock_send([busy, busy, busy]), sleep=no_wait)['status'])
print('404 한 번(다음 응답은 성공):', answer('질문', send=mock_send([missing, done]), sleep=no_wait)['status'])
cut_result = answer('질문', send=mock_send([cut]), sleep=no_wait)
print('잘린 응답:', cut_result['status'], cut_result)

results = [answer(item['문의']) for item in inquiries if item['번호'] in ('Q31', 'Q33', 'Q35', 'Q36', 'Q38')]
for result in results:
    print(f"\n[{result['status']}] {result['question']}\n  → {result['answer'].strip()[:100]}")
print(f"\n요금 합계 {sum(result['cost'] for result in results):.5f}달러, "
      f"실패와 확인 필요 목록 {[(result['question'], result['reason']) for result in results if result['status'] != '완료']}")

429 두 번 뒤 성공: 완료
429 세 번: 실패
404 한 번(다음 응답은 성공): 실패
잘린 응답: 확인 필요 {'question': '질문', 'status': '확인 필요', 'answer': '잘린 답', 'input_tokens': 50, 'output_tokens': 2000, 'cost': 0.001005, 'reason': 'max_tokens'}



[완료] 와이어 본딩 작업 표준 최신본은 어디에 있나요?
  → 와이어 본딩 작업표준의 최신본 위치는 제가 알 수 없습니다. 사내 문서 시스템에 접근할 수 없고, 저에게 제공된 자료에도 해당 정보가 없습니다. 위치를 추측해서 안내하면 오래된 문

[완료] 몰딩 컴파운드 보관 조건이 적힌 문서를 찾고 있습니다.
  → 사내 문서 시스템이나 파일 서버에는 접근할 수 없어서 해당 문서를 직접 찾아드릴 수는 없습니다. 대신 찾는 데 도움이 될 방법을 정리해 드리겠습니다.

**문서를 찾을 수 있는 곳

[완료] 신규 제품 QFN-48의 검사 기준서를 공유해 주세요.
  → 죄송하지만 저는 사내 문서 저장소나 QFN-48 관련 검사 기준서에 접근할 수 없어서, 해당 문서를 직접 공유해 드릴 수는 없습니다. 임의로 기준 수치를 만들어 드리면 실제 라인에

[완료] 3월 품질 회의 회의록을 받을 수 있을까요?
  → 죄송하지만 저는 사내 문서 시스템이나 회의록 저장소에 접근할 수 없어서 3월 품질 회의록을 직접 찾아드릴 수 없습니다. 회의록을 지어내서 안내드리는 것도 곤란하고요.

회의록을 받

[완료] 교대 인수인계 양식 파일이 어디 있는지 알려 주세요.
  → 죄송하지만 사내 파일 저장 위치나 공유 드라이브 경로는 제가 알 수 없어서 정확한 위치를 말씀드릴 수 없습니다. 추측으로 경로를 알려드리면 잘못된 문서를 쓰실 수 있어서 드리지 않

요금 합계 0.00166달러, 실패와 확인 필요 목록 []


## 앞 단계의 코드: 2단계 풀이

In [4]:
import functools

ANSWER_SCHEMA = {'type': 'object', 'additionalProperties': False,
                 'required': ['category', 'urgent', 'needs_document', 'answer'],
                 'properties': {'category': {'type': 'string', 'enum': LABELS}, 'urgent': {'type': 'boolean'},
                                'needs_document': {'type': 'boolean'}, 'answer': {'type': 'string'}}}


def check(data):
    """업무 규칙 검사. 문제가 없으면 None, 있으면 사유 문자열을 돌려줍니다"""
    if data['category'] == '문서와 데이터 요청' and not data['needs_document']:
        return '문서 요청인데 사내 문서가 필요하지 않다고 답함'
    if data['category'] == '안전' and not data['urgent']:
        return '안전 문의인데 긴급이 아니라고 답함'
    return None


def answer_structured(question):
    """1단계의 answer()에 구조화 출력을 더합니다. 결과 dict에 data(파싱한 dict)와 status, reason을 넣습니다"""
    result = answer(question, send=functools.partial(llm.send, schema=ANSWER_SCHEMA))
    result['data'] = None
    if result['status'] != '완료':                   # 잘림과 거절은 스키마에 맞지 않을 수 있으므로 파싱하지 않습니다
        result['status'] = '실패'
        return result
    result['data'] = json.loads(result['answer'])
    problem = check(result['data'])
    result['status'], result['reason'] = ('사람 확인', problem) if problem else ('자동 처리', '')
    return result


for item in [item for item in inquiries if item['번호'] in ('Q21', 'Q24', 'Q31', 'Q43', 'Q49', 'Q56')]:
    result = answer_structured(item['문의'])
    print(f"[{result['status']}] {item['번호']} {item['문의']}\n  → {result['data']}, 사유: {result['reason']}, 요금 {result['cost']:.5f}달러")

[자동 처리] Q21 클린룸 B구역 바닥에 약품이 흘러 있는데 무슨 약품인지 모르겠습니다.
  → {'category': '안전', 'urgent': True, 'needs_document': False, 'answer': '즉시 해당 구역 출입을 통제하고 주변 작업자를 대피시키세요. 흘러나온 물질을 직접 만지거나 흡입하지 말고, 가능하면 자리를 뜨기 전에 환기 설비 상태를 확인하세요. 약품 종류는 제가 사내 정보를 알 수 없어 특정할 수 없습니다. 용기 라벨, 설비 옆 MSDS(물질안전보건자료) 또는 해당 공정 담당자를 통해 확인하시고, 안전관리 담당자 또는 비상 연락처에 즉시 보고해 주세요. 신체에 노출되었다면 바로 의료 조치를 받으세요.'}, 사유: , 요금 0.00019달러


[자동 처리] Q24 비상구 앞에 자재 카트가 쌓여 통로가 막혔습니다.
  → {'category': '안전', 'urgent': True, 'needs_document': False, 'answer': '비상구 앞 통로가 막힌 것은 화재나 비상 시 대피를 방해하는 중대한 안전 위험입니다. 즉시 카트를 비상구 앞에서 치우고 통로를 확보해 주세요. 카트를 치울 수 없다면 주변 작업자에게 알리고 해당 구역 접근을 통제한 뒤, 현장 안전 담당자 또는 관리자에게 바로 보고해 주시기 바랍니다. 자재가 쌓인 원인과 재발 방지 조치는 사내 안전 절차에 따라 처리해야 하며, 구체적인 담당자 연락처나 내부 규정은 제가 알지 못하니 현장 안전 관리 부서에 확인해 주세요.'}, 사유: , 요금 0.00019달러


[자동 처리] Q31 와이어 본딩 작업 표준 최신본은 어디에 있나요?
  → {'category': '문서와 데이터 요청', 'urgent': False, 'needs_document': True, 'answer': "와이어 본딩 작업 표준의 최신본 위치는 제가 사내 문서 저장소 정보를 알지 못해서 확인해 드릴 수 없습니다. 품질 또는 공정기술 담당 부서나 문서관리 시스템에서 '와이어 본딩 작업 표준'으로 검색하여 최신 승인본(개정번호와 시행일 확인)을 찾으시고, 정확한 위치가 필요하면 담당 엔지니어나 문서관리 담당자에게 문의해 주세요."}, 사유: , 요금 0.00016달러


[자동 처리] Q43 다이싱 소 D-01 근처에서 냉각수가 바닥으로 새고 있습니다.
  → {'category': '안전', 'urgent': True, 'needs_document': False, 'answer': '냉각수 누수는 미끄러짐과 감전 위험이 있으므로 즉시 조치가 필요합니다. 우선 해당 구역 접근을 막고 주변 작업자를 대피시키세요. 장비 전원을 차단할 수 있다면 차단하고, 누수 원인을 확인하기 전에는 D-01을 가동하지 마세요. 바닥 누수는 가능하면 흡수재로 처리하고, 즉시 담당 설비 엔지니어와 안전 담당자에게 보고하세요. 사내 비상 절차와 연락처는 제가 알지 못하므로 현장 안전 매뉴얼 또는 담당자에게 확인하시기 바랍니다.'}, 사유: , 요금 0.00019달러


[자동 처리] Q49 마킹 불량 제품이 출하 박스에 섞였는지 확인할 출하 기록이 필요합니다.
  → {'category': '문서와 데이터 요청', 'urgent': True, 'needs_document': True, 'answer': '마킹 불량 제품의 출하 혼입 여부를 확인하려면 해당 로트 또는 박스의 출하 기록이 필요합니다. 확인에 필요한 항목은 출하 일자, 로트 번호, 박스 번호, 제품 시리얼 또는 바코드 정보, 검사 결과 기록입니다. 다만 저는 사내 시스템이나 기록에 접근할 수 없어 실제 출하 내역은 알려드릴 수 없습니다. 품질 또는 생산관리 담당자에게 해당 로트의 출하 기록과 마킹 검사 이력을 요청하고, 의심 박스는 출하 전에 격리해 주시기 바랍니다.'}, 사유: , 요금 0.00019달러


[자동 처리] Q56 본더 한 대가 또 멈췄습니다.
  → {'category': '설비 이상', 'urgent': True, 'needs_document': False, 'answer': "본더 정지 문의로 확인했습니다. 현재 사내 설비 상태 정보와 정지 이력은 제가 알 수 없어 정확한 원인은 말씀드릴 수 없습니다. 우선 해당 본더의 알람 코드와 화면 메시지, 정지 시점, 직전 작업 조건을 확인해 주시고, 안전을 위해 장비 전원과 가동 상태를 점검한 뒤 담당 설비 엔지니어에게 즉시 연락해 주십시오. '또' 멈춘 것이라면 반복 정지 이력도 함께 전달해 주시면 원인 분석에 도움이 됩니다."}, 사유: , 요금 0.00018달러


## 실습 P5. 검색을 도구로 쓰는 에이전트

교재의 절: "관통 프로젝트 5단계"

`answer_agent()`를 작성합니다. 예제 13-3의 `run_agent()`에 오류 결과와 단계 상한을 더하고, 실행 기록(`trace`), 도구 결과에 나온 문서 번호(`seen`), 답에 적힌 문서 번호(`cited`)를 결과 dict에 남깁니다. `cited`에서 `seen`을 뺀 것이 근거 없는 인용입니다. 두 번째 셀이 평가셋의 다섯 문항을 실행하고, 질문을 그대로 한 번 검색했을 때와 에이전트가 정답 문서를 찾았는지 비교합니다. 이 실습은 API를 15~25번 호출합니다.

완료 기준

- 다섯 문항 모두 상태, 단계 수, 실행 기록이 출력됩니다.
- 근거 없는 인용이 0건입니다. 0건이 아닌 답은 사람이 확인할 대상으로 표시합니다.
- 문서에 답이 없는 E57에 찾지 못했다고 답합니다.

<details><summary>힌트 보기</summary>

예제 13-3의 `run_agent()`에서 시작합니다. 도구 결과(`output`)와 답에서 `DOC_ID.findall()`로 문서 번호를 뽑아 집합으로 모으고, 집합의 차(`cited - seen`)로 근거 없는 인용을 구합니다. 모델이 요청할 때마다 `reply.input_tokens + reply.output_tokens`를 더하면 질문 하나에 쓴 토큰이 됩니다.

</details>

In [5]:
import re
from doc_tools import TOOLS, read_doc, search_docs     # 노트북/doc_tools.py: 사내 문서 120건의 검색과 읽기, 도구 정의

AGENT_SYSTEM = ('너는 후공정 회사의 사내 문서 도우미다. 도구로 사내 문서를 찾아 근거가 있는 것만 답한다. '
                '문서에 답이 없으면 찾지 못했다고 답한다. 상태가 폐기인 문서의 값은 쓰지 않는다. '
                '문서끼리 값이 다르면 날짜가 최근인 문서를 따르고 두 문서를 모두 밝힌다. '
                '답의 마지막 줄에 "근거: " 뒤에 근거 문서 번호를 쉼표로 적는다.')
FUNCTIONS = {'search_docs': search_docs, 'read_doc': read_doc}
DOC_ID = re.compile(r'(?<![A-Z])(?:SOP|MAN|QIR|MTG|NOTI|EDU|EHS)-[A-Z0-9]+(?:-\d+)?(?:_R\d+)?')   # 문서 번호


def answer_agent(question, system=AGENT_SYSTEM, functions=FUNCTIONS, max_steps=6, model='소형'):
    """검색을 도구로 쓰는 에이전트. 결과 dict의 키:
    answer 답, status '완료' '사람 확인'(근거 없는 인용) '단계 상한' '실패'(잘림, 거절), steps 모델을 부른 횟수, tokens 입력과 출력 토큰의 합,
    trace 실행 기록 [(도구 이름, 인자, 오류 여부)], seen 도구 결과에 나온 문서 번호,
    cited 답에 적힌 문서 번호, unsupported cited 가운데 seen에 없는 문서 번호(근거 없는 인용)
    """
    result = {'question': question, 'answer': '', 'status': '단계 상한', 'steps': 0, 'tokens': 0,
              'trace': [], 'seen': set()}
    messages = [{'role': 'user', 'content': question}]
    for step in range(1, max_steps + 1):
        reply = llm.ask_messages(messages, system=system, tools=TOOLS, model=model, max_tokens=2000)
        result['steps'] = step
        result['tokens'] += reply.input_tokens + reply.output_tokens
        if reply.stop_reason != 'tool_use':
            result['answer'] = reply.text
            result['status'] = '완료' if reply.stop_reason == 'end' else '실패'
            break
        if step == max_steps:                              # 결과를 돌려줄 다음 요청이 없으므로 도구도 실행하지 않습니다
            break
        messages.append(llm.assistant_turn(reply))
        for call in reply.tool_calls:
            try:
                output, failed = functions[call.name](**call.arguments), False
            except Exception as error:                     # 오류도 결과로 돌려줍니다
                output, failed = f'오류: {error}', True
            result['trace'].append((call.name, call.arguments, failed))
            result['seen'] |= set(DOC_ID.findall(output))
            messages.append(llm.tool_result(call, output, is_error=failed))
    result['cited'] = set(DOC_ID.findall(result['answer']))
    result['unsupported'] = result['cited'] - result['seen']
    if result['status'] == '완료' and result['unsupported']:
        result['status'] = '사람 확인'                     # 근거 없는 인용이 있으면 사람이 원문을 확인합니다
    return result

In [6]:
with open(f'{DATA}/evalset.json', encoding='utf-8') as file:
    evalset = {item['id']: item for item in json.load(file)}

for qid in ['E18', 'E40', 'E41', 'E43', 'E57']:     # 바꿔 말함, 여러 문서, 충돌 둘, 답 없음
    item = evalset[qid]
    result = answer_agent(item['question'])
    gold = set(item['docs'])
    once = set(DOC_ID.findall(search_docs(item['question'])))    # 질문을 그대로 한 번 검색한 결과(API 없음)
    print(f"{qid} [{item['type']}] {item['question']}")
    print(f"  {result['status']}, {result['steps']}단계, {result['tokens']:,}토큰, 도구 {[f'{name}:{list(args.values())[0]}' for name, args, _ in result['trace']]}")
    if gold:
        print(f"  정답 문서 {sorted(gold)}: 한 번 검색 {'찾음' if gold & once else '못 찾음'}, "
              f"에이전트 {'찾음' if gold & result['seen'] else '못 찾음'}, 핵심어 {item['key']} 포함 "
              f"{all(key in result['answer'] for key in item['key'])}")
    print(f"  근거 없는 인용: {sorted(result['unsupported']) or '없음'}")
    print('  답:', result['answer'].strip().replace('\n', ' ')[:200])

E18 [바꿔 말함] 교대 들어가서 처음 하는 점검은 몇 분 안에 끝내야 하나요?
  완료, 3단계, 6,776토큰, 도구 ['search_docs:교대 점검 시간', 'search_docs:교대 후 초물 점검', 'read_doc:SOP-CH-001']
  정답 문서 ['EDU-002', 'SOP-CH-001']: 한 번 검색 못 찾음, 에이전트 찾음, 핵심어 ['30분'] 포함 True
  근거 없는 인용: 없음
  답: 교대를 시작하고 **30분 안에** 교대 시작 점검(SSC)을 끝내야 합니다.  SSC에서는 다음 세 가지를 확인합니다. - 담당 설비의 알람 이력 - 소모품(캐필러리, 블레이드, 컴파운드)의 잔량 - 정전기 손목띠 측정 결과  교육 자료 EDU-002도 SSC를 "교대 시작 후 30분 안에 하는 점검"으로 정의하고 있어 같은 내용입니다.  근거: SOP-


E40 [여러 문서] 고객사 B의 요구로 강화된 기준 두 가지는 무엇이고 각각 얼마인가요?
  완료, 3단계, 9,607토큰, 도구 ['search_docs:고객사 B 강화 기준', 'search_docs:고객사 B 요구 기준', 'read_doc:MTG-20260907', 'read_doc:SOP-VI-001', 'read_doc:SOP-WB-001']
  정답 문서 ['SOP-VI-001', 'SOP-WB-001']: 한 번 검색 찾음, 에이전트 찾음, 핵심어 ['6.0gf', '0.08mm'] 포함 True
  근거 없는 인용: 없음
  답: 고객사 B가 요구해 강화된 기준은 두 가지입니다.  1. **와이어 풀 강도 하한**    - 금선 25μm: 5.0gf에서 **6.0gf 이상**으로 상향    - 구리선 20μm: 6.0gf에서 **7.0gf 이상**으로 상향    - 2026-03-16 개정 3에서 반영되었습니다.  2. **리드 코플래너리티 기준**    - 0.10mm에서 **0.


E41 [충돌] QX48의 몰딩 트랜스퍼 시간은 지금 몇 초인가요?
  완료, 3단계, 9,465토큰, 도구 ['search_docs:QX48 몰딩 트랜스퍼 시간', 'search_docs:QX48 몰딩', 'read_doc:SOP-MD-001', 'read_doc:QIR-2026-019', 'read_doc:MTG-20261116']
  정답 문서 ['MTG-20261116', 'SOP-MD-001']: 한 번 검색 찾음, 에이전트 찾음, 핵심어 ['14초'] 포함 True
  근거 없는 인용: 없음
  답: **QX48 몰딩 트랜스퍼 시간은 문서에 따라 다릅니다. 가장 최근 문서 기준으로는 14초입니다.**  - **MTG-20261116(2026-11-16 품질 회의록)**: QX48 트랜스퍼 시간을 **11월 23일부터 14초**로 바꾸고 SOP-MD-001을 개정하기로 결정했습니다. 14초 평가에서 와이어 스윕 5% 초과 유닛이 0.4%로 줄었습니다. -


E43 [충돌] 11월 테스트 핸들러 정기 PM은 언제 하나요?
  완료, 3단계, 7,523토큰, 도구 ['search_docs:테스트 핸들러 정기 PM', 'search_docs:핸들러 PM 주기', 'read_doc:NOTI-2026-033', 'read_doc:MAN-PM-001']
  정답 문서 ['NOTI-2026-033']: 한 번 검색 찾음, 에이전트 찾음, 핵심어 ['12월 3일'] 포함 True
  근거 없는 인용: 없음
  답: 11월 테스트 핸들러 정기 PM은 **원래 11월 26일(목)** 이었지만, **12월 3일(목)** 으로 미뤄졌습니다.  - 설비보전팀 공지(NOTI-2026-033, 2026-10-05)에서 고객사 B의 연말 물량 증가로 4분기 일부 PM 일정을 바꾼다고 밝혔습니다. - 기준 주기는 설비 PM 주기표(MAN-PM-001)의 "월 1회, 넷째 주 목요일"


E57 [답 없음] 레이저 마킹기의 레이저 파장은 몇 nm인가요?
  완료, 3단계, 7,530토큰, 도구 ['search_docs:레이저 마킹기 파장', 'search_docs:레이저 마킹 nm', 'read_doc:MAN-LM-001', 'read_doc:SOP-MK-001', 'read_doc:QIR-2026-012']
  근거 없는 인용: 없음
  답: 사내 문서에서 레이저 마킹기의 파장 값은 찾지 못했습니다.  확인한 문서는 다음과 같습니다. - **MAN-LM-001 (레이저 마킹기 운용 매뉴얼)**: L-01, L-02가 파이버 레이저라는 것만 나오고 파장 값은 없습니다. - **SOP-MK-001 (레이저 마킹 작업 표준)**: 마킹 내용, 출력 점검, 안전 기준만 있고 파장은 없습니다. - **Q
